# Ohio River: Transport-Engine Case Study

This case study demonstrates the ClearWater-Riverine 2D transport engine on a real-world HEC-RAS 2D model of a reach of the Ohio River. The simulation transports E. coli as a conservative tracer and compares the result against a reference EFDC (Environmental Fluid Dynamics Code) simulation of the same domain. EFDC is a widely used surface-water modeling system developed by the U.S. Environmental Protection Agency, so the comparison demonstrates that the ClearWater-Riverine transport engine reproduces a trusted external solution at production scale.

## What this case study shows

This is a *transport-engine* case study, not a coupled water-quality study. The kinetics are intentionally simple (a single conservative tracer); the demonstration is that ClearWater-Riverine handles a real-world HEC-RAS 2D mesh with realistic boundary forcing and reproduces an independent reference solution.

For coupled water-quality demonstrations, see the Willowbend Creek tutorials in this section.


!!! note "Code cells ship without committed outputs"

    The cells below show the full transport-engine workflow but do not include committed cell outputs because the 110 MB HEC-RAS 2D plan HDF (`OhioRiver_m.p22.hdf`) is not bundled with the documentation. Fetch the HDF as described in `docs/examples/data/ohio_river/README.md` and re-execute the notebook to populate the outputs in place. The animated reference figures in sections 5 and 6 below show what the populated outputs look like.


## Inputs

Inputs live under `docs/examples/data/ohio_river/`:

- `cwr_initial_conditions.csv` -- initial concentration field, one row per HEC-RAS 2D mesh cell
- `cwr_boundary_conditions.csv` -- boundary concentration time series at each inflow
- `efdc/efdc_ohio_river_model_shapefile/` -- EFDC mesh (used for plotting the reference solution)
- `efdc/ohio_river-2010.parquet/` -- EFDC simulated E. coli on the EFDC mesh

!!! note "HEC-RAS plan HDF is fetched separately"

    The HEC-RAS 2D plan HDF (`OhioRiver_m.p22.hdf`, approximately 110 MB) is not committed to this documentation repository. See `docs/examples/data/ohio_river/README.md` for fetch instructions. Without the HDF, the imports and setup cells below will load, but `cwr.ClearwaterRiverine(fpath)` will raise a file-not-found error.


## 1. Imports


In [ ]:
from pathlib import Path

import sys
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import holoviews as hv
import geoviews as gv

import clearwater_riverine as cwr

hv.extension("bokeh")


## 2. Build the transport model from the HEC-RAS 2D plan

ClearWater-Riverine reads mesh topology, face flows, cell volumes, and water-surface elevations directly from the HEC-RAS 2D plan HDF, configures conservative transport for the named constituents in the boundary-condition CSV, and integrates the advection-diffusion equation on the native HEC-RAS unstructured mesh.


In [ ]:
data_dir = Path.cwd().resolve()
while not (data_dir / "docs" / "examples" / "data" / "ohio_river").exists():
    if data_dir.parent == data_dir:
        break
    data_dir = data_dir.parent
data_dir = data_dir / "docs" / "examples" / "data" / "ohio_river"

fpath = data_dir / "OhioRiver_m.p22.hdf"
ic_path = data_dir / "cwr_initial_conditions.csv"
bc_path = data_dir / "cwr_boundary_conditions.csv"

print(f"data dir: {data_dir}")
print(f"HEC-RAS HDF present: {fpath.exists()}")
print(f"initial conditions: {ic_path.exists()}")
print(f"boundary conditions: {bc_path.exists()}")


In [ ]:
ohio = cwr.ClearwaterRiverine(
    str(fpath),
    diffusion_coefficient_input=0.1,
)
ohio.mesh


## 3. Initial and boundary conditions


In [ ]:
ohio.initial_conditions(str(ic_path))
ohio.boundary_conditions(str(bc_path))


## 4. Run the transport solver

On a modern laptop this run takes a few minutes on the bundled mesh.


In [ ]:
%time ohio.update()


## 5. ClearWater-Riverine transport result

The animation below, produced from the simulated `OhioRiver_m.p22` plan, shows E. coli concentration evolving across the Ohio River reach as the upstream pulse is advected and diffused through the mesh. The full simulation covers the same time window as the EFDC reference described in the next section.

![ClearWater-Riverine simulated E. coli on the Ohio River](../../../assets/images/ohio_river/clearwater_riverine_ohio.gif)

*Animated reference figure from the original ClearWater-Riverine Ohio River demonstration. Re-running the notebook with the HEC-RAS plan HDF in place will populate live cell outputs that reproduce this animation.*


## 6. EFDC reference comparison

EFDC was run on its own native mesh of the same Ohio River reach with the same boundary forcing. The two models share boundary conditions and broadly similar mesh extent; their meshes differ in cell shape and resolution because each was constructed in its own modeling environment.

The side-by-side animation below shows the ClearWater-Riverine result on the HEC-RAS 2D mesh (left) and the EFDC reference on its native mesh (right). The two solutions track each other closely in plume shape, peak location, and decay timescale, demonstrating that the ClearWater-Riverine transport engine reproduces the trusted external solution under realistic forcing.

![Side-by-side ClearWater-Riverine and EFDC comparison](../../../assets/images/ohio_river/clearwater_riverine_efdc_comparison.gif)

*Animated reference figure showing ClearWater-Riverine (left) and EFDC (right) on the same time axis.*


## 7. Reproducing this case study

1. Fetch the HEC-RAS plan HDF into `docs/examples/data/ohio_river/` (see the README in that directory; the file lives in the source `ClearWater-riverine` repository).
2. Re-execute this notebook end-to-end. Cells 2--4 build and run the transport model; the live cell outputs will replace the static animations shown in sections 5 and 6.
3. To reproduce the side-by-side EFDC comparison plotting, load the bundled EFDC shapefile and parquet output and overlay them on the ClearWater-Riverine mesh. The original demonstration used `geoviews` and `holoviews.DynamicMap` for an interactive comparison; for static-HTML rendering in the documentation a matplotlib equivalent (one snapshot per representative time) is preferable.


## Next steps

- For coupled water-quality modeling, see [Willowbend Creek Temperature](./02_willowbend_temperature.ipynb) and [Willowbend Creek Nutrients](./03_willowbend_nutrients.ipynb).
- For the transport solver design, read the [Solver](../../user-guide/riverine/solver.md) and [Numerical Methods](../../theory/numerical-methods.md) pages.
- For mass-balance utilities applicable to runs like this one, see the [Post-Processing](../../user-guide/riverine/post-processing.md) page.
